# Step 2: CSV, JSON and XML with Python
**CSE 578: Data Visualization · Olist E-Commerce Project**

Data source: *Brazilian E-Commerce Public Dataset by Olist* (Kaggle), stored in `data/olist/`.
If the folder is empty, download it from the project root with:

```
kaggle datasets download -d olistbr/brazilian-ecommerce -p data/olist --unzip
```

| File | Department | Source table | Columns | Rows |
|---|---|---|---|---|
| `customers.csv` | Sales and Marketplace Operations | customers (+ order value from order_items) | 6 | 200 |
| `orders.csv` | Logistics and Fulfillment | orders | 7 | 300 |

The 200 customers are the buyers of the first 200 sampled orders, so the two files join on `customer_id`.

In [1]:
from pathlib import Path
import json
import xml.etree.ElementTree as ET

import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "step2" else Path.cwd()
RAW_DIR = PROJECT_DIR / "data" / "olist"
OUTPUT_DIR = PROJECT_DIR / "step2" / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

CUSTOMERS_CSV = OUTPUT_DIR / "customers.csv"
ORDERS_CSV = OUTPUT_DIR / "orders.csv"
CUSTOMERS_JSON = OUTPUT_DIR / "customers.json"
CUSTOMERS_XML = OUTPUT_DIR / "customers.xml"

if not RAW_DIR.exists():
    raise FileNotFoundError(f"{RAW_DIR} not found; download the Olist dataset first (see above)")


def print_all(data: pd.DataFrame) -> None:
    """Print every row and column of a DataFrame."""
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 250):
        print(data)

## Task 1: Create two CSV files

**CSV 1: `customers.csv` (Sales and Marketplace Operations)** has 6 columns with different data types and 200 rows:

| Column | Type | Meaning |
|---|---|---|
| `customer_id` | text | Customer key for one order (joins to orders) |
| `customer_unique_id` | text | The same person across repeat orders |
| `customer_zip_code_prefix` | integer | First 5 digits of the postal code |
| `customer_city` | text | City |
| `customer_state` | text (category) | Brazilian state code |
| `order_value` | float | Total paid for the order, price + freight (BRL) |

**CSV 2: `orders.csv` (Logistics and Fulfillment)** has 7 columns and 300 rows. It is the orders table without `order_approved_at`, which is a payment event and belongs to Finance and Payments.

In [2]:
ORDER_ROWS = 300
CUSTOMER_ROWS = 200
RANDOM_SEED = 42

orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")

# Total value of each order = sum of item prices + freight.
items["item_total"] = items["price"] + items["freight_value"]
order_values = items.groupby("order_id")["item_total"].sum().round(2).rename("order_value")

# Sample only orders that have items, so every customer gets an order_value.
orders_with_items = orders[orders["order_id"].isin(order_values.index)]
sampled_orders = (
    orders_with_items.sample(n=ORDER_ROWS, random_state=RANDOM_SEED)
    .sort_values("order_purchase_timestamp")
    .reset_index(drop=True)
)

orders_csv = sampled_orders[[
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]]

customers_csv = (
    sampled_orders.head(CUSTOMER_ROWS)[["customer_id", "order_id"]]
    .merge(customers, on="customer_id", how="left", validate="one_to_one")
    .merge(order_values, left_on="order_id", right_index=True, how="left")
)[[
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
    "order_value",
]]

if customers_csv.isna().any().any():
    raise ValueError("customers.csv has missing values; check the joins with the raw tables")

customers_csv.to_csv(CUSTOMERS_CSV, index=False)
orders_csv.to_csv(ORDERS_CSV, index=False)

print(f"{CUSTOMERS_CSV.name}: {customers_csv.shape[0]} rows x {customers_csv.shape[1]} columns")
print(f"{ORDERS_CSV.name}:    {orders_csv.shape[0]} rows x {orders_csv.shape[1]} columns")

customers.csv: 200 rows x 6 columns
orders.csv:    300 rows x 7 columns


## Task 2: Read both CSV files and print their contents

In [3]:
customers_data = pd.read_csv(CUSTOMERS_CSV)

print(f"CSV 1 - Sales and Marketplace Operations: {CUSTOMERS_CSV.name} {customers_data.shape}")
print("\nColumn data types:")
print(customers_data.dtypes.to_string(), end="\n\n")
print_all(customers_data)

CSV 1 - Sales and Marketplace Operations: customers.csv (200, 6)

Column data types:
customer_id                     str
customer_unique_id              str
customer_zip_code_prefix      int64
customer_city                   str
customer_state                  str
order_value                 float64

                          customer_id                customer_unique_id  customer_zip_code_prefix          customer_city customer_state  order_value
0    51d9009f107df3256dcb608b5cb40a7c  0922f37485310929b1b94e8f0c984ca5                     86037               londrina             PR       156.22
1    950c3e2b1eab245ac9e26cf96a1c6a23  202fad4a680401564558b2cda0c5a8de                     83540      rio branco do sul             PR        33.42
2    a5cd42f5d1eecbf955de47ff1245b5d6  4910006053f92928b62910d568d4ad1e                      5580              sao paulo             SP       613.78
3    bc41a708ec6dabfa8b01320ad60ac6a2  212fb8fd04191789cf67caf8c78e2d1b                     53230     

In [4]:
DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
orders_data = pd.read_csv(ORDERS_CSV, parse_dates=DATE_COLUMNS)

print(f"CSV 2 - Logistics and Fulfillment: {ORDERS_CSV.name} {orders_data.shape}")
print("\nColumn data types:")
print(orders_data.dtypes.to_string(), end="\n\n")
print_all(orders_data)

CSV 2 - Logistics and Fulfillment: orders.csv (300, 7)

Column data types:
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]

                             order_id                       customer_id order_status order_purchase_timestamp order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date
0    ed90ba41c41da31da8e891ff43a584d1  51d9009f107df3256dcb608b5cb40a7c    delivered      2016-10-06 16:43:26          2016-10-11 15:59:10           2016-10-14 15:59:10                    2016-11-28
1    ce9840efc17fbec501472f7c784be1bf  950c3e2b1eab245ac9e26cf96a1c6a23    delivered      2016-10-06 23:21:10          2016-10-21 16:25:50           2016-10-24 16:25:50                    20

## Task 3: Convert `customers.csv` to JSON and XML

- **JSON:** a list of objects, one per customer.
- **XML:** `<customers>` holds one `<customer>` element per row. `parser="etree"` uses Python's built-in XML library, so `lxml` is not needed.

In [5]:
customers_data.to_json(CUSTOMERS_JSON, orient="records", indent=2, force_ascii=False)
customers_data.to_xml(CUSTOMERS_XML, index=False, root_name="customers", row_name="customer", parser="etree")

print(f"Converted {CUSTOMERS_CSV.name} ({len(customers_data)} rows) into {CUSTOMERS_JSON.name} and {CUSTOMERS_XML.name}")

Converted customers.csv (200 rows) into customers.json and customers.xml


## Task 4: Read the JSON and XML files and print their contents

In [6]:
with open(CUSTOMERS_JSON, encoding="utf-8") as file:
    json_records = json.load(file)

print(f"JSON file: {CUSTOMERS_JSON.name} ({len(json_records)} records)")
print("\nFirst record as stored in the file:")
print(json.dumps(json_records[0], indent=2), end="\n\n")

json_data = pd.DataFrame(json_records)
print_all(json_data)

JSON file: customers.json (200 records)

First record as stored in the file:
{
  "customer_id": "51d9009f107df3256dcb608b5cb40a7c",
  "customer_unique_id": "0922f37485310929b1b94e8f0c984ca5",
  "customer_zip_code_prefix": 86037,
  "customer_city": "londrina",
  "customer_state": "PR",
  "order_value": 156.22
}

                          customer_id                customer_unique_id  customer_zip_code_prefix          customer_city customer_state  order_value
0    51d9009f107df3256dcb608b5cb40a7c  0922f37485310929b1b94e8f0c984ca5                     86037               londrina             PR       156.22
1    950c3e2b1eab245ac9e26cf96a1c6a23  202fad4a680401564558b2cda0c5a8de                     83540      rio branco do sul             PR        33.42
2    a5cd42f5d1eecbf955de47ff1245b5d6  4910006053f92928b62910d568d4ad1e                      5580              sao paulo             SP       613.78
3    bc41a708ec6dabfa8b01320ad60ac6a2  212fb8fd04191789cf67caf8c78e2d1b                    

In [7]:
xml_root = ET.parse(CUSTOMERS_XML).getroot()
customer_elements = xml_root.findall("customer")

print(f"XML file: {CUSTOMERS_XML.name} ({len(customer_elements)} <customer> elements)")
print("\nFirst element as stored in the file:")
print(ET.tostring(customer_elements[0], encoding="unicode"))

xml_data = pd.DataFrame([{field.tag: field.text for field in customer} for customer in customer_elements])
# XML stores every value as text, so restore the numeric columns.
xml_data["customer_zip_code_prefix"] = xml_data["customer_zip_code_prefix"].astype(int)
xml_data["order_value"] = xml_data["order_value"].astype(float)
print_all(xml_data)

XML file: customers.xml (200 <customer> elements)

First element as stored in the file:
<customer>
    <customer_id>51d9009f107df3256dcb608b5cb40a7c</customer_id>
    <customer_unique_id>0922f37485310929b1b94e8f0c984ca5</customer_unique_id>
    <customer_zip_code_prefix>86037</customer_zip_code_prefix>
    <customer_city>londrina</customer_city>
    <customer_state>PR</customer_state>
    <order_value>156.22</order_value>
  </customer>
  
                          customer_id                customer_unique_id  customer_zip_code_prefix          customer_city customer_state  order_value
0    51d9009f107df3256dcb608b5cb40a7c  0922f37485310929b1b94e8f0c984ca5                     86037               londrina             PR       156.22
1    950c3e2b1eab245ac9e26cf96a1c6a23  202fad4a680401564558b2cda0c5a8de                     83540      rio branco do sul             PR        33.42
2    a5cd42f5d1eecbf955de47ff1245b5d6  4910006053f92928b62910d568d4ad1e                      5580             

In [8]:
# Check: both converted files hold exactly the same data as the original CSV.
print("JSON matches CSV:", json_data.equals(customers_data))
print("XML matches CSV: ", xml_data.equals(customers_data))

JSON matches CSV: True
XML matches CSV:  True
